# Inicio

In [1]:
#%pip install pandas matplotlib

import pandas as pd
import matplotlib.pyplot as plt

In [2]:
print(pd.__version__)

3.0.3


In [3]:
df = pd.read_csv('server_logs.csv')
print(f'Shape: {df.shape}')

Shape: (5795, 15)


In [4]:
print(f'Info:\n')
df.info()

Info:

<class 'pandas.DataFrame'>
RangeIndex: 5795 entries, 0 to 5794
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   timestamp_event  5795 non-null   str  
 1   received_at      5795 non-null   str  
 2   service_name     5795 non-null   str  
 3   severity         5795 non-null   str  
 4   message          5795 non-null   str  
 5   trace_id         5795 non-null   str  
 6   request_id       5795 non-null   str  
 7   method           5795 non-null   str  
 8   endpoint         5795 non-null   str  
 9   status_code      5795 non-null   int64
 10  latency_ms       5795 non-null   int64
 11  host             5795 non-null   str  
 12  env              5795 non-null   str  
 13  region           5795 non-null   str  
 14  log_type         5795 non-null   str  
dtypes: int64(2), str(13)
memory usage: 679.2 KB


In [5]:
print(f'Dtypes:\n{df.dtypes}')

Dtypes:
timestamp_event      str
received_at          str
service_name         str
severity             str
message              str
trace_id             str
request_id           str
method               str
endpoint             str
status_code        int64
latency_ms         int64
host                 str
env                  str
region               str
log_type             str
dtype: object


In [6]:
print('Priemros 5 registros:\n')
df.head()

Priemros 5 registros:



,timestamp_event,received_at,service_name,severity,message,trace_id,request_id,method,endpoint,status_code,latency_ms,host,env,region,log_type
0,2026-01-10T00:02:39.029160Z,2026-01-10T00:02:39.097160Z,orders-service,INFO,Request completed,bd8e6ebe717b4c43ba5e72e1668641a8,e6b3009adcad,GET,/orders/create,200,99,orders-service-pod-03,prod,sa-east-1,request
1,2026-01-10T00:02:46.081021Z,2026-01-10T00:02:46.196021Z,api-gateway,INFO,Background job completed,40ac5ff97bae43d5b8045484725e0aca,e2aa1cccd1cf,GET,/health,200,122,api-gateway-pod-01,prod,sa-east-1,request
2,2026-01-10T00:04:01.648849Z,2026-01-10T00:04:01.718849Z,notification-service,WARN,Rate limit nearing threshold,c048b378759a4c54a6f7d7251a6acc88,c74106ca8581,POST,/notify/sms,200,646,notification-service-pod-03,prod,sa-east-1,request
3,2026-01-10T00:05:08.148346Z,2026-01-10T00:05:08.236346Z,api-gateway,INFO,Background job completed,072169f708c84227986bfda9f9657bdc,bf10fa3609bd,GET,/checkout,200,127,api-gateway-pod-03,prod,sa-east-1,request
4,2026-01-10T00:05:19.590837Z,2026-01-10T00:05:19.632837Z,inventory-service,INFO,Health check OK,e76a214131d24a4ea1acf389362e34cb,f672d669eda7,GET,/inv/release,200,144,inventory-service-pod-02,prod,sa-east-1,request


In [7]:
# TRansformamos el timestamp_event a datetime y nos aseguramos de que se transforme correctamente.
df["timestamp_event"] = pd.to_datetime(df["timestamp_event"])
df.dtypes

timestamp_event    datetime64[us, UTC]
received_at                        str
service_name                       str
severity                           str
message                            str
trace_id                           str
request_id                         str
method                             str
endpoint                           str
status_code                      int64
latency_ms                       int64
host                               str
env                                str
region                             str
log_type                           str
dtype: object

# Exploración inicial.

### ¿Cuántos logs hay en total?

In [8]:
df.shape[0]

5795

### En total hay 5795 logs

### ¿Qué severidad aparece más?

In [9]:
sever = df['severity']
sever.value_counts()

severity
INFO        3542
WARN        1358
ERROR        775
CRITICAL     120
Name: count, dtype: int64

### La severidad que más aparece es INFO.

### ¿Qué servicio genera más logs?

In [10]:
df['service_name'].value_counts()

service_name
api-gateway             1509
orders-service          1057
inventory-service        964
payment-service          842
auth-service             778
notification-service     645
Name: count, dtype: int64

### El servicio que genera más logs es api-gateway.

### ¿Qué servicio genera menos logs?

### El servicio que genera menos logs es notification-service.

### ¿Cuál es el mensaje más repetido?

In [11]:
print(df['message'].value_counts())

message
Health check OK                                   1196
Background job completed                          1185
Request completed                                 1161
Order creation failed - inventory lock timeout     197
Rate limit nearing threshold                       193
Elevated 401 responses at gateway                  193
Retrying upstream request                          184
Slow response detected                             183
Cannot send receipt - Payment pending              170
Inventory endpoint degraded - high latency         114
Payment gateway unavailable                        103
Database deadlock detected                          99
Inventory reservation retry                         98
Checkout failed - upstream payment timeout          88
Increased 5xx responses detected at gateway         82
Order created but payment pending                   71
Suspicious login rate detected                      70
Possible credential stuffing detected               69
Pa

### El mensaje más repetido es "Health check OK".

### ¿Cuál es el mensaje “malo” más repetido?

In [12]:
bad_event = df[ (df["severity"] == "ERROR") | (df["severity"] == "CRITICAL") | (df["status_code"] >= 500)]
bad_event['message'].value_counts()

message
Order creation failed - inventory lock timeout    197
Payment gateway unavailable                       103
Database deadlock detected                         99
Checkout failed - upstream payment timeout         88
Possible credential stuffing detected              69
Payment Gateway Timeout - 504 Gateway Time-out     67
Database query failed                              62
External dependency error                          59
Invalid credentials - 401 spike                    54
Upstream timeout                                   46
Out of memory                                      29
Service crash detected                             22
Name: count, dtype: int64

### El Bad Event más repetido es:
#### Order creation failed - inventory lock timeout
#### Con 197 apariciones.

In [ ]:
df['is_bad'] = ((df["severity"] == "ERROR") | (df["severity"] == "CRITICAL") | (df["status_code"] >= 500))

windows = df.groupby(pd.Grouper(key='timestamp_event', freq='5min')).agg( total_events=('timestamp_event', 'size'), bad_events=('is_bad', 'sum')).reset_index()

windows['bad_rate'] = windows['bad_events'] / windows['total_events']

windows.head()

,timestamp_event,total_events,bad_events,bad_rate
0,2026-01-10 00:00:00+00:00,3,0,0.000000
1,2026-01-10 00:05:00+00:00,4,0,0.000000
2,2026-01-10 00:10:00+00:00,3,0,0.000000
3,2026-01-10 00:15:00+00:00,4,0,0.000000
4,2026-01-10 00:20:00+00:00,9,1,0.111111


In [ ]:
windows[windows['total_events'] >=20].sort_values('bad_rate', ascending=False).head()

,timestamp_event,total_events,bad_events,bad_rate
134,2026-01-10 11:10:00+00:00,189,110,0.582011
135,2026-01-10 11:15:00+00:00,228,129,0.565789
136,2026-01-10 11:20:00+00:00,111,59,0.531532
463,2026-01-11 14:35:00+00:00,255,117,0.458824
462,2026-01-11 14:30:00+00:00,156,68,0.435897


In [ ]:
critical_window = windows[windows['total_events'] >= 20].sort_values('bad_rate', ascending=False).iloc[0]
critical_window

timestamp_event    2026-01-10 11:10:00+00:00
total_events                             189
bad_events                               110
bad_rate                            0.582011
Name: 134, dtype: object

In [73]:
inicio = critical_window['timestamp_event']
fin = inicio + pd.Timedelta('5min')

critical_df = df[(df['timestamp_event'] >= inicio) & (df['timestamp_event'] < fin)]
critical_df.head()

,timestamp_event,received_at,service_name,severity,message,trace_id,request_id,method,endpoint,status_code,latency_ms,host,env,region,log_type,is_bad
689,2026-01-10 11:10:01.262893+00:00,2026-01-10T11:10:01.351893Z,payment-service,ERROR,External dependency error,ed89807b9c854e9eb0412a250c180bfc,8022182c05d9,GET,/pay/status,502,388,payment-service-pod-03,prod,sa-east-1,request,True
690,2026-01-10 11:10:03.851393+00:00,2026-01-10T11:10:04.528393Z,orders-service,ERROR,Order creation failed - inventory lock timeout,e06c4ed1570f46efa88fde74338d9edd,9cac9bb3d67e,POST,/orders/create,504,1816,orders-service-pod-03,prod,sa-east-1,incident,True
691,2026-01-10 11:10:03.856393+00:00,2026-01-10T11:10:04.347393Z,api-gateway,WARN,Inventory endpoint degraded - high latency,e06c4ed1570f46efa88fde74338d9edd,9cac9bb3d67e,POST,/checkout,200,625,api-gateway-pod-01,prod,sa-east-1,incident,False
692,2026-01-10 11:10:04.134393+00:00,2026-01-10T11:10:05.160393Z,inventory-service,ERROR,Database deadlock detected,e06c4ed1570f46efa88fde74338d9edd,9cac9bb3d67e,GET,/inv/reserve,504,860,inventory-service-pod-02,prod,sa-east-1,incident,True
693,2026-01-10 11:10:06.291494+00:00,2026-01-10T11:10:07.521494Z,orders-service,ERROR,Order creation failed - inventory lock timeout,51fe17ae89b94c3bb7db10e6a85479b3,2ef9ba521d80,POST,/orders/cancel,504,1538,orders-service-pod-01,prod,sa-east-1,incident,True


In [87]:
critical_df_bad = critical_df[critical_df['is_bad'] == True]
critical_df_bad.shape

(110, 17)

In [88]:
critical_df_bad['service_name'].value_counts()

service_name
orders-service       72
inventory-service    37
payment-service       1
Name: count, dtype: int64

In [89]:
critical_df_bad['message'].value_counts()

message
Order creation failed - inventory lock timeout    72
Database deadlock detected                        37
External dependency error                          1
Name: count, dtype: int64

In [90]:
critical_df_bad['endpoint'].value_counts()

endpoint
/orders/cancel    26
/orders/create    25
/orders/status    21
/inv/reserve      18
/inv/stock        13
/inv/release       6
/pay/status        1
Name: count, dtype: int64

# Top 3 message más repetidos:

## Order creation failed - inventory lock timeout    72
### Database deadlock detected                        37
#### External dependency error                          1

# Top 5 endpoint más repetidos:

## /orders/cancel    26 
### /orders/create    25 
#### /orders/status    21 
/inv/reserve      18  
/inv/stock        13

In [93]:
server_down = critical_df_bad[critical_df_bad['status_code'] >= 500]
server_down.shape
critical_df_bad['status_code'].value_counts()

status_code
504    86
502    10
500     9
503     5
Name: count, dtype: int64

## Vemos que todos los BAD_EVENTS del momento critico tienen status_code mayor o igual a 500,  
## lo cual indica que todos fueron por errores del servidor.